# 04 - Negotiation & Coordination
**Agentic X | Studio 67** — Real components only. Run from the repository root.


## 1. Project Setup

Connect to the Agentic X repository and verify the required project files.

In [1]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/ProjectsSpaceHub/AgenticX.git"
ROOT = Path("/content/AgenticX")

if not ROOT.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(ROOT)],
        check=True
    )
else:
    if not (ROOT / ".git").exists():
        raise RuntimeError("Project folder is not a Git repository")

    changes = subprocess.run(
        ["git", "-C", str(ROOT), "status", "--porcelain"],
        capture_output=True,
        text=True,
        check=True
    ).stdout.strip()

    if not changes:
        subprocess.run(
            ["git", "-C", str(ROOT), "pull", "--ff-only"],
            check=True
        )
    else:
        print("Local changes detected. Skipping pull.")

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

required_files = [
    "core/models.py",
    "core/environment.py",
    "core/communication.py",
    "core/negotiation.py",
    "agents/creative.py",
    "agents/finance.py",
    "agents/safety.py",
    "agents/operations.py"
]

for file in required_files:
    assert (ROOT / file).is_file(), f"Missing: {file}"

print("GitHub project connected successfully!")
print("All required files found!")
print("Project path:", ROOT)

GitHub project connected successfully!
All required files found!
Project path: /content/AgenticX


### 2. Dependencies & Agent Imports

Install dependencies and import the four agents and shared components.

In [2]:
import subprocess
import sys

# Install project dependencies
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "-q", "-r", str(ROOT / "requirements.txt")
    ],
    check=True
)

# Shared components
from core.environment import load_dataset, get_case
from core.models import AgentEvaluation, ACLMessage, RunResult
from core.communication import MessageBus
from core.negotiation import NegotiationEngine

# Four independent agents
from agents.creative import CreativeAgent
from agents.finance import FinanceAgent
from agents.safety import SafetyAgent
from agents.operations import OperationsAgent

# Verify dataset
cases_df, options_df = load_dataset()

assert len(cases_df) == 150
assert len(options_df) == 450

print("All dependencies installed!")
print("All four agents imported successfully!")
print("Communication and negotiation modules ready!")
print("Dataset:", len(cases_df), "cases,", len(options_df), "options")

All dependencies installed!
All four agents imported successfully!
Communication and negotiation modules ready!
Dataset: 150 cases, 450 options


### 3. Gemini Configuration & Agent Initialization

Configure Gemini securely and initialize the four real agents.

In [3]:
import os
from getpass import getpass
from google.colab import userdata

# Load Gemini API key securely
try:
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass("Enter your Gemini API key: ")

if not api_key.strip():
    raise ValueError("Gemini API key is required.")

os.environ["GOOGLE_API_KEY"] = api_key.strip()
os.environ["AGENTICX_MODEL"] = "gemini-3.8-flash"

del api_key

# Initialize real agents
creative_agent = CreativeAgent()
finance_agent = FinanceAgent()
safety_agent = SafetyAgent()
operations_agent = OperationsAgent()

agents = [
    creative_agent,
    finance_agent,
    safety_agent,
    operations_agent
]

assert len(agents) == 4
assert len({agent.name for agent in agents}) == 4

print("Gemini configured successfully!")
print("All four agents initialized!")

for agent in agents:
    print("✓", agent.name)

Enter your Gemini API key: ··········
Gemini configured successfully!
All four agents initialized!
✓ Creative
✓ Finance
✓ Safety
✓ Operations


### 4. Negotiation Rules & Scenario Selection

Define negotiation constraints and inspect the available solutions.

In [4]:
import pandas as pd
from IPython.display import display

# Negotiation settings
MAX_ROUNDS = 3
REQUIRE_ALL_APPROVALS = True

# Select a real scenario
case, options = get_case("SCR-001")

assert len(options) == 3
assert len({o["option_id"] for o in options}) == 3

print("Case:", case["case_id"])
print("Incident:", case["incident"])
print("Maximum negotiation rounds:", MAX_ROUNDS)
print("All four agents must approve:", REQUIRE_ALL_APPROVALS)

display(
    pd.DataFrame(options)[
        ["option_id", "option_name", "cost_sar", "duration_min"]
    ]
)

Case: SCR-001
Incident: Main LED screen failed during final rehearsal
Maximum negotiation rounds: 3
All four agents must approve: True


,option_id,option_name,cost_sar,duration_min
0,SCR-001-O1,Use backup LED,1290,40
1,SCR-001-O2,Rent replacement screen,5920,70
2,SCR-001-O3,Continue with projection,1210,25


### 5. Multi-Round Negotiation Engine

Upgrade the coordinator to handle agent conflicts, alternative proposals, and human escalation.

In [5]:
from pathlib import Path

# Locate the current negotiation engine
negotiation_path = ROOT / "core" / "negotiation.py"

assert negotiation_path.exists(), "Negotiation file not found"

# Read the existing implementation
original_code = negotiation_path.read_text(
    encoding="utf-8"
)

assert "class NegotiationEngine" in original_code
assert "def run(" in original_code

# Save a backup outside the repository
backup_path = Path("/content/negotiation_original.py")

if not backup_path.exists():
    backup_path.write_text(
        original_code,
        encoding="utf-8"
    )

print("Current negotiation engine verified!")
print("Original implementation backed up!")
print("Backup path:", backup_path)
print("Ready to upgrade the negotiation engine.")

Current negotiation engine verified!
Original implementation backed up!
Backup path: /content/negotiation_original.py
Ready to upgrade the negotiation engine.


### 5.1. Adaptive Proposal Selection

Prioritize alternative solutions based on agent feedback and negotiation constraints.

In [6]:
import math

negotiation_source = '''
"""Adaptive multi-agent negotiation for Studio 67."""

import math
from core.communication import MessageBus
from core.models import RunResult


class NegotiationEngine:

    def __init__(
        self, creative, finance, safety,
        operations, max_rounds=3
    ):
        self.agents = [
            creative, finance, safety, operations
        ]

        if not isinstance(max_rounds, int) or max_rounds < 1:
            raise ValueError("max_rounds must be positive")

        self.max_rounds = max_rounds

    @staticmethod
    def _number(value, default=0.0):
        try:
            number = float(value)
            return number if math.isfinite(number) else default
        except (TypeError, ValueError):
            return default

    @staticmethod
    def _is_true(value):
        return (
            value is True
            or value == 1
            or (
                isinstance(value, str)
                and value.strip().lower() == "true"
            )
        )

    def _priority(self, option, objections):
        """Rank options using previous agent objections."""

        cost = self._number(option.get("cost_sar"))
        duration = self._number(option.get("duration_min"))
        quality = self._number(option.get("quality_score"))
        savings = self._number(
            option.get("gross_savings_sar")
        )

        safety_ready = all([
            self._is_true(option.get("safety_ok")),
            self._is_true(option.get("permit_ok")),
            self._is_true(
                option.get("mitigation_verified")
            )
        ])

        risk = self._number(
            option.get("risk_score_after_mitigation"),
            default=999
        )

        if "Safety" in objections:
            return (
                int(safety_ready),
                -risk,
                savings - cost,
                quality,
                -duration
            )

        if "Finance" in objections:
            return (
                savings - cost,
                quality,
                -duration,
                int(safety_ready),
                -risk
            )

        if "Operations" in objections:
            return (
                -duration,
                quality,
                savings - cost,
                int(safety_ready),
                -risk
            )

        return (
            quality,
            -duration,
            savings - cost,
            int(safety_ready),
            -risk
        )
'''

# Validate syntax before modifying the actual Python file
compile(
    negotiation_source,
    "negotiation_stage_1",
    "exec"
)

print("Adaptive proposal selection code validated!")
print("Original negotiation.py remains unchanged.")
print("Ready for the multi-round negotiation logic.")

Adaptive proposal selection code validated!
Original negotiation.py remains unchanged.
Ready for the multi-round negotiation logic.


### 5.2. Agent Evaluation & ACL Responses

Collect real agent evaluations and record their decisions using ACL messages.

In [7]:
# Add the shared evaluation model
negotiation_source = negotiation_source.replace(
    "from core.models import RunResult",
    "from core.models import RunResult, AgentEvaluation"
)

# Add the agent evaluation method
negotiation_source += '''
    def _evaluate_proposal(
        self, case, option, round_number, bus
    ):
        """Collect real evaluations from all four agents."""

        evaluations = []

        for agent in self.agents:

            # Coordinator requests an evaluation
            request = bus.send(
                round_number,
                "Coordinator",
                agent.name,
                "REQUEST",
                f"Evaluate option {option['option_id']}"
            )

            # Real agent evaluation
            evaluation = agent.evaluate(case, option)

            # Verify the returned evaluation
            if not isinstance(evaluation, AgentEvaluation):
                raise TypeError(
                    f"{agent.name} returned an invalid evaluation"
                )

            if evaluation.agent != agent.name:
                raise ValueError("Agent name mismatch")

            if evaluation.option_id != option["option_id"]:
                raise ValueError("Option ID mismatch")

            evaluations.append(evaluation)

            # Share evaluation reasoning
            bus.send(
                round_number,
                agent.name,
                "Coordinator",
                "INFORM",
                evaluation.reason,
                in_reply_to=request.message_id
            )

            # Record approval or refusal
            performative = (
                "ACCEPT" if evaluation.approved else "REFUSE"
            )

            bus.send(
                round_number,
                agent.name,
                "ALL",
                performative,
                f"{option['option_id']}: {evaluation.reason}"
            )

        return evaluations
'''

# Validate Python syntax
compile(
    negotiation_source,
    "negotiation_stage_2",
    "exec"
)

print("Agent evaluation logic validated!")
print("All four agents included!")
print("ACL responses configured!")
print("Original negotiation.py remains unchanged.")

Agent evaluation logic validated!
All four agents included!
ACL responses configured!
Original negotiation.py remains unchanged.


### 5.3. Multi-Round Negotiation & Conflict Resolution

Evaluate one proposal per round, adapt to agent objections, and escalate unresolved conflicts.

In [8]:
negotiation_source += '''
    def run(self, case, options):
        """Run adaptive negotiation across multiple rounds."""

        bus = MessageBus(case["case_id"])
        remaining = list(options)
        all_evaluations = []
        objections = set()
        rounds_used = 0

        # Prevent duplicate proposals
        option_ids = [o["option_id"] for o in remaining]
        if len(option_ids) != len(set(option_ids)):
            raise ValueError("Duplicate option IDs")

        for round_number in range(1, self.max_rounds + 1):

            if not remaining:
                break

            rounds_used = round_number

            # Announce a new negotiation round
            bus.send(
                round_number,
                "Coordinator",
                "ALL",
                "CFP",
                f"Request proposals for round {round_number}"
            )

            # Select an alternative based on previous objections
            proposal = max(
                remaining,
                key=lambda o: self._priority(o, objections)
            )

            remaining.remove(proposal)

            bus.send(
                round_number,
                "Coordinator",
                "ALL",
                "PROPOSE",
                f"Proposed option: {proposal['option_id']}"
            )

            # Collect real decisions from all four agents
            evaluations = self._evaluate_proposal(
                case, proposal, round_number, bus
            )

            all_evaluations.extend(evaluations)

            rejected = [
                e for e in evaluations
                if not e.approved
            ]

            # Agreement requires approval from all four agents
            if not rejected:

                bus.send(
                    round_number,
                    "Coordinator",
                    "ALL",
                    "ACCEPT",
                    f"Agreement reached: {proposal['option_id']}"
                )

                return RunResult(
                    case_id=case["case_id"],
                    status="AGREED",
                    selected_option_id=proposal["option_id"],
                    rounds=round_number,
                    evaluations=all_evaluations,
                    messages=bus.messages,
                    reason="All four agents approved the solution"
                )

            # Learn which agents objected
            objections = {
                evaluation.agent
                for evaluation in rejected
            }

            bus.send(
                round_number,
                "Coordinator",
                "ALL",
                "INFORM",
                "Proposal rejected by: "
                + ", ".join(sorted(objections))
            )

        # No acceptable proposal was found
        bus.send(
            max(1, rounds_used),
            "Coordinator",
            "Human Manager",
            "REQUEST",
            "Human escalation: no solution received "
            "approval from all four agents."
        )

        return RunResult(
            case_id=case["case_id"],
            status="ESCALATED",
            selected_option_id=None,
            rounds=rounds_used,
            evaluations=all_evaluations,
            messages=bus.messages,
            reason="No acceptable solution within the negotiation limit"
        )
'''

# Check syntax without modifying negotiation.py
compile(
    negotiation_source,
    "negotiation_stage_3",
    "exec"
)

print("Multi-round negotiation logic validated!")
print("Adaptive alternative selection configured!")
print("Agreement and escalation rules configured!")
print("Ready for integration testing.")

Multi-round negotiation logic validated!
Adaptive alternative selection configured!
Agreement and escalation rules configured!
Ready for integration testing.


### 5.4. Negotiation Engine Validation

Load the upgraded engine and verify adaptive proposal selection.

In [9]:
# Load the upgraded engine temporarily
namespace = {}

exec(
    compile(negotiation_source, "negotiation_candidate", "exec"),
    namespace
)

CandidateNegotiationEngine = namespace["NegotiationEngine"]

candidate_engine = CandidateNegotiationEngine(
    creative_agent,
    finance_agent,
    safety_agent,
    operations_agent,
    max_rounds=MAX_ROUNDS
)

# Test proposal selection under different objections
for objections in [
    set(),
    {"Finance"},
    {"Safety"},
    {"Operations"}
]:
    selected = max(
        options,
        key=lambda o: candidate_engine._priority(
            o, objections
        )
    )

    print(
        "Objections:", sorted(objections) or "None",
        "| Selected:", selected["option_id"],
        "|", selected["option_name"]
    )

# Ensure the original file remains unchanged
assert negotiation_path.read_text(
    encoding="utf-8"
) == original_code

print("\nAdaptive selection test passed!")
print("Original negotiation.py is still unchanged.")

Objections: None | Selected: SCR-001-O1 | Use backup LED
Objections: ['Finance'] | Selected: SCR-001-O3 | Continue with projection
Objections: ['Safety'] | Selected: SCR-001-O1 | Use backup LED
Objections: ['Operations'] | Selected: SCR-001-O3 | Continue with projection

Adaptive selection test passed!
Original negotiation.py is still unchanged.


### 5.5. Real Agent Evaluation Test

Evaluate a real proposal using all four agents and verify ACL responses.

In [10]:
# Create a communication bus for the selected case
test_bus = MessageBus(case["case_id"])

# Select the first real proposal
test_proposal = options[0]

print("Testing proposal:", test_proposal["option_name"])
print("Collecting real agent evaluations...\n")

# Evaluate using all four real agents
test_evaluations = candidate_engine._evaluate_proposal(
    case,
    test_proposal,
    round_number=1,
    bus=test_bus
)

# Display agent decisions
evaluation_df = pd.DataFrame([
    evaluation.model_dump()
    for evaluation in test_evaluations
])

display(evaluation_df)

# Validate evaluations and communication
assert len(test_evaluations) == 4
assert len(test_bus.messages) == 12

assert {
    evaluation.agent
    for evaluation in test_evaluations
} == {
    "Creative", "Finance", "Safety", "Operations"
}

assert all(
    evaluation.option_id == test_proposal["option_id"]
    for evaluation in test_evaluations
)

print("\nReal agent evaluation completed!")
print("ACL messages:", len(test_bus.messages))
print("All evaluation checks passed!")

Testing proposal: Use backup LED



,agent,option_id,approved,score,reason,blocking
0,Creative,SCR-001-O1,True,89.0,The solution replaces the failed screen using ...,False
1,Finance,SCR-001-O1,True,100.0,Committed plus option cost: 1290.00 SAR versus...,True
2,Safety,SCR-001-O1,True,100.0,All safety checks passed,True
3,Operations,SCR-001-O1,True,95.0,The option is highly feasible operationally. E...,True



Real agent evaluation completed!
ACL messages: 12
All evaluation checks passed!


### 5.6. Direct Agreement Test

Execute a real negotiation and verify the final decision and ACL messages.

In [11]:
# Execute the complete negotiation process
direct_result = candidate_engine.run(case, options)

print("Case:", direct_result.case_id)
print("Status:", direct_result.status)
print("Selected Option:", direct_result.selected_option_id)
print("Rounds:", direct_result.rounds)
print("Evaluations:", len(direct_result.evaluations))
print("ACL Messages:", len(direct_result.messages))

# Validate the result
assert direct_result.is_mock is False
assert 1 <= direct_result.rounds <= MAX_ROUNDS

if direct_result.status == "AGREED":
    selected_evaluations = [
        e for e in direct_result.evaluations
        if e.option_id == direct_result.selected_option_id
    ]

    assert len(selected_evaluations) == 4
    assert all(e.approved for e in selected_evaluations)

    print("\nAgreement verified successfully!")

else:
    print("\nNo agreement reached. Review agent objections.")

# Display actual ACL communication
messages_df = pd.DataFrame([
    message.model_dump()
    for message in direct_result.messages
])

display(
    messages_df[
        ["round", "sender", "receiver", "performative", "content"]
    ]
)

print("\nFull negotiation execution completed!")

Case: SCR-001
Status: AGREED
Selected Option: SCR-001-O1
Rounds: 1
Evaluations: 4
ACL Messages: 15

Agreement verified successfully!


,round,sender,receiver,performative,content
0,1,Coordinator,ALL,CFP,Request proposals for round 1
1,1,Coordinator,ALL,PROPOSE,Proposed option: SCR-001-O1
2,1,Coordinator,Creative,REQUEST,Evaluate option SCR-001-O1
3,1,Creative,Coordinator,INFORM,The solution directly resolves the main LED sc...
4,1,Creative,ALL,ACCEPT,SCR-001-O1: The solution directly resolves the...
5,1,Coordinator,Finance,REQUEST,Evaluate option SCR-001-O1
6,1,Finance,Coordinator,INFORM,Committed plus option cost: 1290.00 SAR versus...
7,1,Finance,ALL,ACCEPT,SCR-001-O1: Committed plus option cost: 1290.0...
8,1,Coordinator,Safety,REQUEST,Evaluate option SCR-001-O1
9,1,Safety,Coordinator,INFORM,All safety checks passed



Full negotiation execution completed!


### 6. Conflict Scenario Preparation

Inspect real agent objections and identify suitable scenarios for multi-round negotiation.

In [12]:
# Preview real conflicts without calling Gemini
conflict_preview = []

for case_id in ["BDG-001", "SFT-001"]:

    test_case, test_options = get_case(case_id)

    ranked_options = sorted(
        test_options,
        key=lambda o: candidate_engine._priority(o, set()),
        reverse=True
    )

    for rank, option in enumerate(ranked_options, start=1):

        finance = finance_agent.evaluate(test_case, option)
        safety = safety_agent.evaluate(test_case, option)

        conflict_preview.append({
            "case_id": case_id,
            "proposal_order": rank,
            "option_id": option["option_id"],
            "finance_approved": finance.approved,
            "safety_approved": safety.approved,
            "objections": ", ".join([
                agent.agent
                for agent in [finance, safety]
                if not agent.approved
            ]) or "None"
        })

conflict_df = pd.DataFrame(conflict_preview)

display(conflict_df)

assert len(conflict_df) == 6

print("Conflict scenario preview completed!")

,case_id,proposal_order,option_id,finance_approved,safety_approved,objections
0,BDG-001,1,BDG-001-O3,False,True,Finance
1,BDG-001,2,BDG-001-O2,True,True,None
2,BDG-001,3,BDG-001-O1,True,True,None
3,SFT-001,1,SFT-001-O3,True,True,None
4,SFT-001,2,SFT-001-O2,True,True,None
5,SFT-001,3,SFT-001-O1,True,False,Safety


Conflict scenario preview completed!


### 6.1. Multi-Round Conflict Resolution Test

Test real negotiation after a financial objection.

In [13]:
# Run a real budget conflict scenario
budget_case, budget_options = get_case("BDG-001")

budget_result = candidate_engine.run(
    budget_case,
    budget_options
)

print("Case:", budget_result.case_id)
print("Status:", budget_result.status)
print("Selected Option:", budget_result.selected_option_id)
print("Rounds:", budget_result.rounds)

# Show agent decisions across negotiation rounds
budget_evaluations_df = pd.DataFrame([
    e.model_dump()
    for e in budget_result.evaluations
])

display(
    budget_evaluations_df[
        ["option_id", "agent", "approved", "score", "reason"]
    ]
)

# Verify the real Finance objection
first_option = budget_evaluations_df["option_id"].iloc[0]

assert first_option == "BDG-001-O3"

assert (
    (budget_evaluations_df["option_id"] == first_option)
    & (budget_evaluations_df["agent"] == "Finance")
    & (~budget_evaluations_df["approved"])
).any()

assert budget_result.rounds >= 2

print("\nFinancial conflict detected successfully!")
print("Negotiation continued beyond round 1!")

Case: BDG-001
Status: AGREED
Selected Option: BDG-001-O2
Rounds: 2


,option_id,agent,approved,score,reason
0,BDG-001-O3,Creative,True,99.0,"The solution keeps the premium package, delive..."
1,BDG-001-O3,Finance,False,0.0,Projected spend after action: 29176.00 SAR ver...
2,BDG-001-O3,Safety,True,100.0,All safety checks passed
3,BDG-001-O3,Operations,True,92.0,Operationally feasible: execution time of 20 m...
4,BDG-001-O2,Creative,True,82.0,The standard AV package maintains strong event...
5,BDG-001-O2,Finance,True,100.0,Projected spend after action: 20686.00 SAR ver...
6,BDG-001-O2,Safety,True,100.0,All safety checks passed
7,BDG-001-O2,Operations,True,95.0,Operationally feasible: execution time of 50 m...



Financial conflict detected successfully!
Negotiation continued beyond round 1!


### 6.2. Human Escalation Test

Verify that unresolved conflicts are escalated after three negotiation rounds.

In [14]:
# Load the safety conflict scenario
safety_case, safety_options = get_case("SFT-001")

# Controlled stress test: no execution time remains
escalation_case = safety_case.copy()
escalation_case["time_left_min"] = 0

# Run real negotiation
escalation_result = candidate_engine.run(
    escalation_case,
    safety_options
)

print("Case:", escalation_result.case_id)
print("Status:", escalation_result.status)
print("Rounds:", escalation_result.rounds)
print("Selected Option:", escalation_result.selected_option_id)
print("Reason:", escalation_result.reason)

# Validate escalation
assert escalation_result.status == "ESCALATED"
assert escalation_result.selected_option_id is None
assert escalation_result.rounds == 3
assert escalation_result.is_mock is False

# Verify human escalation message
human_messages = [
    message
    for message in escalation_result.messages
    if message.receiver == "Human Manager"
]

assert len(human_messages) == 1
assert human_messages[0].performative == "REQUEST"

# Display agent refusals
refusals = [
    message.model_dump()
    for message in escalation_result.messages
    if message.performative == "REFUSE"
]

display(pd.DataFrame(refusals)[
    ["round", "sender", "content"]
])

print("\nHuman escalation test passed!")

Case: SFT-001
Status: ESCALATED
Rounds: 3
Selected Option: None
Reason: No acceptable solution within the negotiation limit


,round,sender,content
0,1,Operations,SFT-001-O3: Operational constraints failed: time
1,2,Creative,SFT-001-O1: The proposed option leaves the eme...
2,2,Safety,"SFT-001-O1: Safety veto: safety_ok, permit_ok,..."
3,2,Operations,SFT-001-O1: Operational constraints failed: time
4,3,Operations,SFT-001-O2: Operational constraints failed: time



Human escalation test passed!


### 7. Final Validation & Engine Integration

Validate negotiation outcomes and save the upgraded Python engine.

In [15]:
import importlib
import inspect

# Verify all three negotiation outcomes
assert direct_result.status == "AGREED"
assert direct_result.rounds == 1

assert budget_result.status == "AGREED"
assert budget_result.rounds == 2

assert escalation_result.status == "ESCALATED"
assert escalation_result.rounds == 3
assert escalation_result.selected_option_id is None

# Verify agreement and message integrity
for result in [
    direct_result,
    budget_result,
    escalation_result
]:
    assert result.is_mock is False

    message_ids = [
        message.message_id
        for message in result.messages
    ]

    assert len(message_ids) == len(set(message_ids))

    if result.status == "AGREED":
        selected = [
            evaluation
            for evaluation in result.evaluations
            if evaluation.option_id == result.selected_option_id
        ]

        assert len(selected) == 4
        assert all(e.approved for e in selected)

print("All negotiation scenarios validated!")

# Validate Python source
compile(
    negotiation_source,
    str(negotiation_path),
    "exec"
)

# Save upgraded engine locally
negotiation_path.write_text(
    negotiation_source,
    encoding="utf-8"
)

# Reload the actual Python module
import core.negotiation as negotiation_module

importlib.reload(negotiation_module)

NegotiationEngine = negotiation_module.NegotiationEngine

# Verify the upgraded module
assert Path(
    inspect.getfile(NegotiationEngine)
).resolve() == negotiation_path.resolve()

assert hasattr(NegotiationEngine, "_priority")
assert hasattr(NegotiationEngine, "_evaluate_proposal")

print("Upgraded negotiation.py saved successfully!")
print("Python module reloaded successfully!")
print("File:", negotiation_path)

All negotiation scenarios validated!
Upgraded negotiation.py saved successfully!
Python module reloaded successfully!
File: /content/AgenticX/core/negotiation.py


### 8. System Integration Check

Verify that the upgraded negotiation engine integrates with the project runner.

In [16]:
import importlib
import core.runner as runner_module
import core.negotiation as negotiation_module

# Reload the project runner
importlib.reload(runner_module)

# Build the integrated engine
integrated_engine = runner_module.build_engine()

# Verify integration
assert isinstance(
    integrated_engine,
    negotiation_module.NegotiationEngine
)

assert len(integrated_engine.agents) == 4
assert integrated_engine.max_rounds == 3

assert {
    agent.name for agent in integrated_engine.agents
} == {
    "Creative", "Finance",
    "Safety", "Operations"
}

assert negotiation_path.read_text(
    encoding="utf-8"
) == negotiation_source

print("Project runner integration: PASS")
print("Four real agents connected: PASS")
print("Upgraded negotiation engine: PASS")
print("Notebook 05 compatibility: PASS")

Project runner integration: PASS
Four real agents connected: PASS
Upgraded negotiation engine: PASS
Notebook 05 compatibility: PASS


### 9. Export Negotiation Engine

Download the upgraded negotiation module for GitHub integration.

In [17]:
from google.colab import files

# Verify the final Python file
assert negotiation_path.exists()

assert negotiation_path.read_text(
    encoding="utf-8"
) == negotiation_source

print("Final negotiation engine verified!")
print("Downloading negotiation.py...")

files.download(str(negotiation_path))

Final negotiation engine verified!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>